In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from pathlib import Path
import glob,math,re,gc
from transformers import AutoTokenizer, AutoModel

device = "cuda" if torch.cuda.is_available() else "cpu"
n_gpus = torch.cuda.device_count()
print(f"device: {device}, GPUs available: {n_gpus}")
if device == "cuda":
    print(f"GPU 0: {torch.cuda.get_device_name(0)}")

def show_gpu_mem(tag=""):
    if device == "cuda":
        for i in range(n_gpus):
            alloc = torch.cuda.memory_allocated(i) / 1e9
            reserved = torch.cuda.memory_reserved(i) / 1e9
            print(f"[{tag}] GPU {i}: {alloc:.2f} GB allocated, {reserved:.2f} GB reserved")

def free_memory():
    gc.collect()
    torch.cuda.empty_cache()

show_gpu_mem("startup")

device: cuda, GPUs available: 2
GPU 0: Tesla T4
[startup] GPU 0: 0.00 GB allocated, 0.00 GB reserved
[startup] GPU 1: 0.00 GB allocated, 0.00 GB reserved


In [ ]:
def load_data():
    path = glob.glob("/kaggle/input/**/*.txt",recursive=True)
    data_path = Path(path[0])
    with open(data_path,mode="r") as f:
        data = f.read()
        length = (0.02* len(data))
        data = data[:int(length)]
        print(f"Length of dataset in characters: {len(data)}")
    split = (0.8 * len(data))
    train_data,test_set = data[:int(split)],data[int(split):]
    subset = (0.25 * len(test_set))
    test_data,valid_data = test_set[int(subset):],test_set[:int(subset)]
    return train_data,test_data,valid_data

train_data,test_data,valid_data = load_data()
print(len(train_data),len(test_data),len(valid_data))

Length of dataset in characters: 2986527
2389221 447980 149326


### chunking

In [3]:
def chunk_data(data,chunk_size = 5000):
    words = re.findall(r"[A-Za-z']+",data)
    chunks = [" ".join(words[i:i+chunk_size]) for i in range(0,len(words),chunk_size)]
    return chunks

train_chunk = chunk_data(train_data)
test_chunk = chunk_data(test_data)
valid_chunk = chunk_data(valid_data)
print(len(train_chunk),len(test_chunk),len(valid_chunk))
print(train_chunk[0])
for i in range(3):
    print(f"Train chunk {i+1}: {train_chunk[i][:100]}...")
    print(f"Test chunk {i+1}: {test_chunk[i][:100]}...")
    print(f"Valid chunk {i+1}: {valid_chunk[i][:100]}...")

86 16 6
MARCH All Stories New and Complete Publisher Editor IF is published bi monthly by Quinn Publishing Company Inc Kingston New York Volume No Copyright by Quinn Publishing Company Inc Application for Entry' as Second Class matter at Post Office Buffalo New York pending Subscription for issues in U S and Possessions Canada for issues elsewhere Aiiow four weeks for change of address All stories appearing in this magazine are fiction Any similarity to actual persons is coincidental c a fcopy Printed ia U S A A chat with the editor i science fiction magazine called IF The title was selected after much thought because of its brevity and on the theory it is indicative of the field and will be easy to remember The tentative title that just morning and couldn't remember it until we'd had a cup of coffee it was summarily discarded A great deal of thought and effort lias gone into the formation of this magazine We have had the aid of several very talented and generous people for which we ar

## tokenization

### we tokenize batches per trainig loop to avoid crashing

In [4]:
tokenizer = AutoTokenizer.from_pretrained("xlm-roberta-base")

config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [9]:
def tokenize_data(data,tokenizer,max_length=512):
    all_input_ids,all_mask_attention = [],[]
    for chunk in data:
        encoding = tokenizer(chunk,
                             truncation=True,
                             padding="max_length",
                             max_length=max_length,
                             return_tensors = "pt",
                             )
        all_input_ids.append(encoding["input_ids"])
        all_mask_attention.append(encoding["attention_mask"])
    return {"input_ids": torch.cat(all_input_ids,dim = 0), "attention_mask": torch.cat(all_mask_attention,dim = 0)}

probed_data = tokenize_data(test_chunk,tokenizer)
length = probed_data["input_ids"].sum(dim = 1)
print(f"Length of probed data: {length.max().item()}")
print(f"mean: {length.float().mean()}, var: {length.float().var()}")
print(probed_data["input_ids"].shape,probed_data["attention_mask"].shape)
print(probed_data["input_ids"][0],probed_data["attention_mask"][0])

Length of probed data: 13062872
mean: 11013660.0, var: 1819647410176.0
torch.Size([16, 512]) torch.Size([16, 512])
tensor([     0,   5299,   8966,   5809,    186, 124632,    360,  98848,    442,
           509,     70,  34376,     70,  32834,   1970,      7,    136, 113518,
          4295,    136,   6117,   3134,      7,    136,   3853, 162882,   1314,
          2750,   3542,  34376,    360,  84740,    642,  14037,   5045,    111,
            70,   5701,    360,   9942,    442,    509,    756,     70,  80097,
           136,    110,  22729, 124666,    136, 105416,  69674,      7,    400,
            90,    136, 105416,    400,     90, 169463,    136,      6,  66157,
          1830,    360,  10271,    442,    509,  49002,    581,  76746,    450,
         43334,   1755,     70,   8999,    459,  28016,  43334,    525,     23,
         10271,    581,     10, 192592,    450,  40158,      7,     70,   8999,
            83,  25277,   5928,  29367,     23,  10271,     62,   3395,    678,
     

In [10]:
encoded_train_data = tokenize_data(train_chunk,tokenizer)
encoded_test_data = tokenize_data(test_chunk,tokenizer)
encoded_valid_data = tokenize_data(valid_chunk,tokenizer)

encoded_train_data = {k:v.cpu() for k,v in encoded_train_data.items()}
encoded_test_data = {k:v.cpu() for k,v in encoded_test_data.items()}
encoded_valid_data = {k:v.cpu() for k,v in encoded_valid_data.items()}
print(encoded_train_data["input_ids"].shape,encoded_test_data["input_ids"].shape,encoded_valid_data["input_ids"].shape)

torch.Size([86, 512]) torch.Size([16, 512]) torch.Size([6, 512])


#### positional encoding for a model to know what a sequence or sentence means.

In [11]:
class positional_encoding(nn.Module):
    def __init__(self,embedded_dim, max_length = 512,dropout = 0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        pe = torch.zeros(max_length,embedded_dim)
        position = torch.arange(0,max_length).unsqueeze(1)
        div_term = torch.exp(torch.arange(0,embedded_dim,2)*(-math.log(10000.0)/embedded_dim))
        pe[:,0::2] = torch.sin(position*div_term)
        pe[:,1::2] = torch.cos(position*div_term)
        self.register_buffer('pe',pe.unsqueeze(0))

    def forward(self,x):
        x = x+self.pe[:,:x.size(1)]
        return self.dropout(x) 

show_gpu_mem("after positional encoding")       

[after positional encoding] GPU 0: 0.00 GB allocated, 0.00 GB reserved
[after positional encoding] GPU 1: 0.00 GB allocated, 0.00 GB reserved


# transformer

# building a multi-head attention
`q,k,v` -->query,key,value , which are `W(Q,K,V)` ,here W is weights

In [12]:
class Multihead_attention(nn.Module):
    def __init__(self,embedded_dim,nheads,dropout =0.3):
        super(Multihead_attention,self).__init__()

        self.embedded_dim = embedded_dim
        self.nheads = nheads
        self.head_dim = embedded_dim//nheads
# linear layer for q,k,v
        self.w_q = nn.Linear(embedded_dim,embedded_dim)
        self.w_k = nn.Linear(embedded_dim,embedded_dim)
        self.w_v = nn.Linear(embedded_dim,embedded_dim)
#output projection
        self.w_o = nn.Linear(embedded_dim,embedded_dim)
        self.dropout = nn.Dropout(dropout)
        self.scale = math.sqrt(self.head_dim)

    def forward(self,query,key,value,mask =None):
        Q = self.w_q(query)
        K = self.w_k(key)
        V = self.w_v(value)   

        #split inti multiple heads 
        Q= Q.view(Q.size(0),Q.size(1),self.nheads,self.head_dim).transpose(1,2)
        K = K.view(K.size(0),K.size(1),self.nheads,self.head_dim).transpose(1,2)
        V = V.view(V.size(0),V.size(1),self.nheads,self.head_dim).transpose(1,2)

        scores = torch.matmul(Q,K.transpose(-2,-1))/self.scale
        if mask is not None:
            scores = scores.masked_fill(mask==0,float('-inf'))
        attention_weights = torch.softmax(scores,dim=-1)
        attention_output = torch.matmul(self.dropout(attention_weights),V)

        attention_output = attention_output.transpose(1,2).contiguous().view(attention_output.size(0), -1, self.embedded_dim)
        output = self.w_o(attention_output)
        return output,attention_weights

In [13]:
attention_mask = encoded_train_data['attention_mask']
print(attention_mask.shape)

torch.Size([86, 512])


In [18]:
mha = Multihead_attention(embedded_dim= 512,nheads=8)
batch_size = 2
input_ids = encoded_train_data['input_ids'][:batch_size]
embeddings = nn.Embedding(tokenizer.vocab_size,512)
positional_enc = positional_encoding(embedded_dim=512)
x = embeddings(input_ids)
x = positional_enc(x)
query = key = value = x
mask = attention_mask[:batch_size, None, None, :] 
output,attention_weights = mha(query,key,value,mask = mask)
print(f"Output shape: {output.shape}, Attention weights shape: {attention_weights.shape}")
show_gpu_mem("after multihead attention")

Output shape: torch.Size([2, 512, 512]), Attention weights shape: torch.Size([2, 8, 512, 512])
[after multihead attention] GPU 0: 0.00 GB allocated, 0.00 GB reserved
[after multihead attention] GPU 1: 0.00 GB allocated, 0.00 GB reserved


## building a encoder which contains:
`multi-head attention`,`add-norm layer`,`feed forward network`,`add-norm layer`

In [19]:
class transformer_encoder(nn.Module):
    def __init__(self,input_dim,hidden_dim,n_heads,dropout):
        super(transformer_encoder,self).__init__()
        self.multihead_attention = Multihead_attention(embedded_dim=input_dim,nheads=n_heads,dropout=dropout)
        self.norm1 = nn.LayerNorm(input_dim)
        self.dropout1 = nn.Dropout(dropout)

        self.ff = nn.Sequential(
            nn.Linear(input_dim,hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim,input_dim),
        )
        self.norm2 = nn.LayerNorm(input_dim)
        self.dropout2 = nn.Dropout(dropout)

    def forward(self,x,mask = None):
        attention_output,_ = self.multihead_attention(x,x,x,mask)
        x = self.norm1(x + self.dropout1(attention_output))
        ff_output = self.ff(x)
        x = self.norm2(x +self.dropout2(ff_output))   
        return x 

## building a decoder:
`self-attention`,`cross attention`,`ff`

In [ ]:
class transformer_decoder(nn.Module):
    def __init__(self,input_dim,hidden_dim,n_heads,dropout):
        super(transformer_decoder,self).__init__()
        self.multihead_attention = Multihead_attention(input_dim,n_heads,dropout)
        self.norm1 = nn.LayerNorm(input_dim)
        self.dropout1 = nn.Dropout(dropout)
        self.cross_attention = Multihead_attention(input_dim,n_heads,dropout)
        self.norm2 = nn.LayerNorm(input_dim)
        self.dropout2 = nn.Dropout(dropout)

        self.ff = nn.Sequential(
            nn.Linear(input_dim,hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim,input_dim),
        )
        self.norm3 = nn.LayerNorm(input_dim)
        self.dropout3 = nn.Dropout(dropout)

    def forward(self,x,encoder_output,causal_mask = None,cross_mask =None):
        attention_output,_ = self.multihead_attention(x,x,x,causal_mask)
        x = self.norm1(x+self.dropout1(attention_output))
        cross_output,_ = self.cross_attention(x,encoder_output,encoder_output,cross_mask)
        x = self.norm2(x+self.dropout2(cross_output))
        ff_out = self.ff(x)
        x = self.norm3(x+self.dropout3(ff_out))
        return x


In [ ]:
def causal_mask(seq_length,device):
    mask = torch.tril(torch.ones(seq_length,seq_length,device=device)).bool()
    return mask[None,None,:,:]
seq_length = x.size(1)
causal_masks = causal_mask(seq_length,device=device)

# transformer architecture

In [ ]:
class Transformer(nn.Module):
    def __init__(self, vocab_size,embedded_dim = 512,n_heads = 8,hidden_dim = 1024,
                 n_encoder_layers = 4,n_decoder_layers =4,max_length = 512,dropout = 0.1):
        super().__init__()
        self.src_embedding = nn.Embedding(vocab_size,embedded_dim)
        self.target_embedding = nn.Embedding(vocab_size,embedded_dim)
        self.positional_encode =positional_encoding(embedded_dim,max_length,dropout=0.1)

        self.encoder_block = nn.ModuleList([transformer_encoder(embedded_dim,hidden_dim,n_heads,dropout) for _ in range(n_encoder_layers)])
        self.decoder_block = nn.ModuleList([transformer_decoder(embedded_dim,hidden_dim,n_heads,dropout) for _ in range(n_decoder_layers)])

        self.output_projection = nn.Linear(embedded_dim,vocab_size)

    def encode(self,src_ids,src_mask = None):
        x = self.src_embedding(src_ids)
        x = self.positional_encode(x)
        for layer in self.encoder_block:
            x = layer(x,src_mask)
        return x

    def decode(self,target_ids,encode_output,src_mask = None):
        x = self.target_embedding(target_ids)
        x = self.positional_encode(x)
        seq_length = x.size(1)
        causal = causal_mask(seq_length,x.device)
        for layer in self.decoder_block:
            x = layer(x,encode_output,causal_mask = causal,cross_maask = src_mask)
        return x

    def forward(self,src_ids,target_ids,src_mask = None):    
        encode_output = self.encode(src_ids,src_mask)
        decode_output = self.decode(target_ids,encode_output,src_mask)
        logits = self.output_projection(decode_output)
        return logits

In [ ]:
model = Transformer(vocab_size=tokenizer.vocab_size).to(device)
src_ids = encoded_train_data["input_ids"][:batch_size].to(device)
src_mask = attention_mask[:batch_size, None, None, :].to(device)
target_ids = src_ids.to(device)
logits = model(src_ids,target_ids,src_mask)
print(logits.shape)